# Supply Chain Demand Agent — Project Walkthrough

This notebook walks through every component of this project from start to finish.
It's written for someone who knows Python and basic ML but is seeing tools like
TFT, RAG, and agentic AI for the first time.

**Before running this notebook:**
- Make sure you're using the `Supply Chain Agent (Python 3.12)` kernel
  (Kernel → Change Kernel → Supply Chain Agent)
- Set your Anthropic API key in the cell below
- Run cells in order from top to bottom

---

## What we're building

A company like Applied Materials manages thousands of spare parts for
semiconductor equipment. Knowing which parts will run out — and when — is
critical. A stockout can halt a production line and cost millions.

This project builds an AI agent that:
1. **Forecasts** 30-day demand per part using a Temporal Fusion Transformer
2. **Answers questions** using a RAG pipeline over supply chain documents
3. **Acts autonomously** — deciding which tools to call to answer a question

---

## Architecture at a glance

```
User question
     ↓
[Agent — Claude] decides which tools to call
     ↓                    ↓                    ↓
[Inventory tool]   [Forecast tool]     [RAG search tool]
 reads CSV         runs TFT model      searches ChromaDB
     ↓                    ↓                    ↓
                [Claude synthesizes answer]
                          ↓
               Specific, data-grounded response
```

## Setup — API key and imports

We load the Anthropic API key from the `.env` file at the project root.
That file is in `.gitignore` so it never gets uploaded to GitHub.

To set your key: open `.env` in the project root and replace `your-key-here`
with your actual Anthropic API key. That's the only place you ever touch it.

In [ ]:
import sys
import os
from dotenv import load_dotenv

# Add the project root to Python's search path so we can import our modules
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')  # Work from the project root so file paths resolve correctly

# Load the API key from the .env file at the project root.
# .env is listed in .gitignore - it will never be committed to GitHub.
# To set your key: open .env and replace 'your-key-here' with your actual key.
load_dotenv('.env')

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
if api_key and api_key != 'your-key-here':
    print('API key loaded from .env')
else:
    print('No API key found. Open .env and add your Anthropic API key.')

print('Working directory:', os.getcwd())

---
## Part 1: The Dataset

### What's in the data?

We generate synthetic (fake but realistic) supply chain data.
Real supply chain data from companies is confidential, so building
synthetic data with realistic patterns is standard practice.

Each row represents one part on one day:
- `part_id` — which spare part (PART_001 through PART_050)
- `date` — the calendar date
- `demand` — how many units were requested that day
- `inventory` — current stock on hand
- `category` — type of part (Valve, Sensor, Pump, Controller, Filter)
- `supplier` — which supplier makes it
- `region` — where it's deployed
- `lead_time_days` — how long reordering takes
- `price_usd` — unit cost

This is a **multivariate time series**: 50 separate time series
(one per part), each with its own demand history and attributes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Load the dataset (run data/generate_data.py first if this fails)
df = pd.read_csv('data/supply_chain_data.csv', parse_dates=['date'])

print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Unique parts: {df["part_id"].nunique()}')
print()
df.head(10)

In [ ]:
# Let's look at the demand pattern for a few parts
# You should see: upward trend, seasonal waves, and occasional spikes

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for ax, part in zip(axes, ['PART_001', 'PART_015', 'PART_030']):
    part_data = df[df['part_id'] == part].set_index('date')
    ax.plot(part_data.index, part_data['demand'], linewidth=0.8, alpha=0.9)
    ax.set_title(f"{part} — {part_data['category'].iloc[0]} from {part_data['supplier'].iloc[0]}")
    ax.set_ylabel('Daily demand (units)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.tight_layout()
plt.suptitle('Demand patterns for three example parts', y=1.01, fontsize=13)
plt.show()

# What you're seeing:
# - The wavy pattern = yearly seasonality (peaks ~October)
# - The upward drift = slow demand growth over 4 years
# - The sharp vertical spikes = random demand surges (emergency orders)

In [ ]:
# Quick summary: what does our dataset look like across all parts?

print('Parts by category:')
print(df.drop_duplicates('part_id')['category'].value_counts())

print('\nParts by supplier:')
print(df.drop_duplicates('part_id')['supplier'].value_counts())

print('\nDemand statistics per part (daily):')
print(df.groupby('part_id')['demand'].agg(['mean', 'std', 'max']).describe().round(1))

---
## Part 2: The Temporal Fusion Transformer

### Why TFT and not LSTM?

You already know how an LSTM works: it reads one time step at a time,
passes information forward through a hidden state, and tries to learn
patterns in the sequence.

TFT is an improvement in three specific ways:

**1. It separates inputs into types**

| Input type | Example | How TFT uses it |
|---|---|---|
| Static (never changes) | part category, supplier | Learns "Valves from SupplierC behave like X" |
| Past unknown | demand, inventory | Learns from historical patterns |
| Future known | month, day of week | Uses calendar to predict seasonality ahead of time |

An LSTM treats all of these the same. TFT processes each type differently.

**2. Variable Selection Network**

Before forecasting, TFT learns which input features matter most for each part.
If `day_of_week` turns out to be useless for Valves, TFT learns to ignore it.
This is automatic feature selection built into the architecture.

**3. Self-attention over the full history**

Like in GPT/BERT, TFT uses attention to look back across the entire
encoder window and decide which past time steps are most relevant now.
An LSTM's memory fades — TFT can jump back 90 days and find a relevant spike.

**4. Prediction intervals, not just a single number**

TFT outputs p10, p50, and p90 quantiles:
- p10: demand will probably be above this (lower bound)
- p50: best single estimate (median)
- p90: demand will probably be below this (use this for safety stock)

This is what makes TFT useful for inventory decisions — you can say
"order enough for the p90 scenario" rather than betting on one number.

In [ ]:
# Load and prepare the data for TFT
# This adds the time index and calendar features TFT needs

from forecasting.model import load_and_prepare, build_dataset, build_model, ENCODER_LENGTH, DECODER_LENGTH

df_prepared = load_and_prepare('data/supply_chain_data.csv')

print('Added columns:', [c for c in df_prepared.columns if c not in df.columns])
print('\nSample of prepared data:')
df_prepared[['part_id', 'date', 'time_idx', 'month', 'day_of_week', 'quarter', 'demand']].head(8)

In [ ]:
# Build the training and validation datasets
# This creates the sliding window samples TFT trains on

print(f'Encoder length (look-back window): {ENCODER_LENGTH} days')
print(f'Decoder length (forecast horizon): {DECODER_LENGTH} days')
print()

training_ds, validation_ds = build_dataset(df_prepared)

print(f'Training samples: {len(training_ds):,}')
print(f'Validation samples: {len(validation_ds):,}')
print()
print('What one training sample looks like:')
print('  Input: 90 days of demand + inventory + calendar features')
print('  Target: the next 30 days of demand (what we want to predict)')
print()
print('TFT will learn to map the 90-day history → 30-day forecast')
print('using ALL 50 parts simultaneously, sharing knowledge across them.')

In [ ]:
# Inspect the model architecture
import torch

model = build_model(training_ds)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'TFT model created.')
print(f'Trainable parameters: {n_params:,}')
print(f'GPU available: {torch.cuda.is_available()}')
print()
print('For context: GPT-2 has 117M parameters. Our TFT has', n_params // 1000, 'K.')
print('Smaller = faster to train, still powerful for this task.')

### Training the model

The cell below runs the full training pipeline including MLflow logging.
**This takes 5–20 minutes** depending on your hardware (faster on GPU).

Skip this cell if you just want to see the other project components.
The agent and RAG work independently of whether the model is trained.

After training:
- Run `mlflow ui` in your terminal
- Open `http://localhost:5000`
- You'll see the experiment with all metrics, parameters, and the model artifact

In [ ]:
# OPTIONAL: Train the TFT model
# Remove the comment below to run training

# from forecasting.train import train
# best_checkpoint = train()
# print(f'Training complete. Best model: {best_checkpoint}')

print('Training skipped. Uncomment the lines above to train.')
print('The forecast tool will use a statistical baseline until the model is trained.')

---
## Part 3: RAG — Retrieval-Augmented Generation

### The problem RAG solves

Claude is trained on general internet data. It has no idea about:
- Your company's reorder policies
- Which suppliers have had reliability issues
- Past stockout incidents and what caused them
- Your safety stock calculation formula

RAG gives Claude a searchable memory of your own documents.

### How it works

**Step 1 — Ingest (one-time setup):**
```
Document text → embedding model → vector [0.21, -0.45, 0.87, ...] → stored in ChromaDB
```

**Step 2 — Retrieve (at query time):**
```
Query text → same embedding model → query vector
ChromaDB: find stored vectors closest to query vector
Return top-3 most relevant document texts
```

**Step 3 — Generate:**
```
Prompt to Claude: "Given these documents: [retrieved docs]\nAnswer: [user question]"
Claude answers using the retrieved context
```

### What are embeddings?

An embedding converts text into a list of numbers that captures the **meaning**.
Sentences with similar meanings get similar vectors.

```
'inventory is running low'  → [0.21, -0.45, 0.87, ...]
'parts are almost out'      → [0.19, -0.43, 0.85, ...]  ← similar!
'the weather is sunny'      → [-0.62, 0.11, -0.34, ...] ← different
```

We use `all-MiniLM-L6-v2` — a small, fast model that produces 384-dimensional vectors.
It runs locally, no API key needed.

In [ ]:
# Build the knowledge base (embeds all 10 documents into ChromaDB)
# On first run this downloads the embedding model (~80MB)

from rag.ingest import build_knowledge_base, DOCUMENTS

print(f'We have {len(DOCUMENTS)} documents in our knowledge base:')
for doc in DOCUMENTS:
    print(f"  [{doc['category']:10s}] {doc['id']}")

print()
collection = build_knowledge_base()
print(f'\nChromaDB collection has {collection.count()} stored documents.')

In [ ]:
# Test retrieval with different queries
# Watch which documents come back and how the similarity scores change

from rag.retriever import retrieve, format_context

queries = [
    'reorder policy for valves',
    'SupplierC delay problems',
    'how to calculate safety stock',
    'what happened during the 2022 stockout',
]

for q in queries:
    print(f'Query: "{q}"')
    results = retrieve(q, top_k=2)
    for r in results:
        print(f'  [{r["similarity"]:.3f}] {r["id"]} ({r["category"]})')
    print()

In [ ]:
# See the full formatted context that gets sent to Claude

query = 'what is the reorder policy when inventory is low?'
docs = retrieve(query)
context = format_context(docs)

print('CONTEXT SENT TO CLAUDE:')
print('='*60)
print(context[:1500], '...' if len(context) > 1500 else '')

---
## Part 4: The Agent

### What makes it an "agent" and not just a chatbot?

A regular chatbot takes your question and produces a response directly.

An agent follows a **ReAct loop**:
1. **Reason** — what do I need to answer this?
2. **Act** — call a tool to get the data
3. **Observe** — look at the tool result
4. **Reason again** — do I need more data?
5. Repeat until ready to answer
6. **Answer** — synthesize everything into a response

### Our three tools

| Tool | What it does |
|---|---|
| `search_knowledge_base` | Searches ChromaDB for relevant policies/supplier info/incidents |
| `get_inventory_status` | Reads the CSV, computes days of supply, flags at-risk parts |
| `get_demand_forecast` | Runs TFT model (or baseline) to forecast next 30 days |

Claude decides **which tools to call and in what order** based on the question.
We don't hardcode the logic — Claude figures it out from the tool descriptions.

In [ ]:
# Test the individual tools directly (no API key needed for these)

from agent.agent import run_tool

print('=== TOOL: get_inventory_status (top 5 at-risk) ===')
result = run_tool('get_inventory_status', {'top_n': 5})
print(result)

In [ ]:
print('=== TOOL: get_demand_forecast for PART_007 ===')
result = run_tool('get_demand_forecast', {'part_id': 'PART_007'})
print(result)

In [ ]:
print('=== TOOL: search_knowledge_base ===')
result = run_tool('search_knowledge_base', {'query': 'SupplierC quality issues'})
print(result[:600], '...')

In [ ]:
# Now run the full agent loop (requires ANTHROPIC_API_KEY)
# Watch how Claude chains multiple tool calls to build its answer

from agent.agent import run_agent

if os.environ.get('ANTHROPIC_API_KEY', '').startswith('sk-ant'):
    question = "Which parts are most at risk right now, and what actions should I take?"
    print(f'Question: {question}')
    print('='*60)
    answer = run_agent(question)
    print(answer)
else:
    print('Set ANTHROPIC_API_KEY at the top of this notebook to run the agent.')
    print('Tools work without an API key - only the Claude synthesis step needs it.')

---
## Part 5: MLOps with MLflow

### What is MLOps?

MLOps (Machine Learning Operations) is the set of practices for taking
an ML model from a notebook to reliable production use.

Key concerns:
- **Experiment tracking** — which model version performed best?
- **Reproducibility** — can someone recreate this exact model?
- **Model versioning** — what changed between v1 and v2?
- **Deployment** — how do we serve predictions reliably?
- **Monitoring** — is the model still accurate after deployment?

### What MLflow does in this project

Every time you run `train.py`, MLflow automatically records:
- All hyperparameters (learning rate, hidden size, dropout, etc.)
- Metrics at every epoch (training loss, validation loss, MAE)
- The final trained model file
- How many training samples were used

You can compare multiple runs side-by-side and see exactly
which settings produced the best model.

After running training, start the MLflow UI:
```bash
mlflow ui
# Open http://localhost:5000
```

In [ ]:
# Check if any MLflow runs exist from training
import mlflow

mlflow.set_tracking_uri('mlruns')

try:
    experiment = mlflow.get_experiment_by_name('supply-chain-tft')
    if experiment:
        runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
        if len(runs) > 0:
            print(f'Found {len(runs)} training run(s):')
            cols = ['run_id', 'start_time', 'metrics.val_mae_p50', 'metrics.epochs_trained']
            available = [c for c in cols if c in runs.columns]
            print(runs[available].to_string(index=False))
        else:
            print('No training runs yet. Run training to see results here.')
    else:
        print('No MLflow experiment found. Run training first.')
except Exception as e:
    print('No MLflow data found yet. Train the model first.')

---
## Part 6: Running the Full App

All the components above are brought together in `app.py`.

To run the app locally:
```bash
# From the project root, activate the venv first:
venv\Scripts\activate   (Windows)
source venv/bin/activate (Mac/Linux)

# Then run:
streamlit run app.py
```

Open http://localhost:8501 in your browser.

### To deploy for free on Streamlit Cloud:
1. Push this repo to GitHub
2. Go to [share.streamlit.io](https://share.streamlit.io)
3. Connect your GitHub account
4. Select this repo and `app.py` as the entry point
5. Add your `ANTHROPIC_API_KEY` in the Secrets section
6. Click Deploy — you get a public URL

Note: Streamlit Cloud free tier does not support GPU,
so the app will use the statistical forecast baseline.
The RAG and agent components work fully in the cloud.

---
## Summary

Here's what this project demonstrates and why each part matters for the resume:

| Component | Technology | JD Requirement |
|---|---|---|
| Demand forecasting | TFT (pytorch-forecasting) | "material forecast, demand forecast" |
| Deep learning framework | PyTorch + PyTorch Lightning | "PyTorch" explicitly mentioned |
| Agentic AI | Anthropic Claude tool use | "agents, reasoning workflows" |
| RAG pipeline | sentence-transformers + ChromaDB | "embedding-based search systems" |
| LLM application | Anthropic API | "LLM projects" |
| MLOps | MLflow tracking + checkpointing | "model deployment, versioning, monitoring" |
| Stakeholder UI | Streamlit | "AI-driven prototypes" |
| Data engineering | Pandas, synthetic data generation | "data mining", "data processing" |

The project is end-to-end: from raw data → trained model → production-style
agent with a deployable UI. That end-to-end ownership is what differentiates
this from a notebook experiment.